# Pure MoS2 vs MoS2+hBN, epsilon=20

This notebook overlays the pure monolayer MoS2 run against the dielectric-environment MoS2+hBN run. It focuses on the same CNT-style observables: current, EDOS, retarded polarization, screened interaction magnitude, and retarded self-energy.


In [ ]:
from pathlib import Path
import re
import tomllib
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (8.5, 4.8),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "legend.frameon": False,
})

def find_w90_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for parent in (start, *start.parents):
        if (parent / "data_analysis").is_dir() and (parent / "mos2").is_dir():
            return parent
    raise RuntimeError("Could not find w90 root.")

W90_ROOT = find_w90_root()
DATA_ANALYSIS_ROOT = W90_ROOT / "data_analysis"
ROOT = W90_ROOT / "mos2" / "dielectric_environment"
PURE = ROOT.parent / "monolayer" / "outputs_mos2_epsilon_20"
HBN = ROOT / "epsilon_20" / "outputs_mos2_hbn_24iter_fresh"
PURE_CONFIG = ROOT.parent / "monolayer" / "quatrex_config.toml"
HBN_CONFIG = ROOT / "quatrex_config_mos2_hbn_epsilon_20.toml"
PURE_LOG = ROOT.parent / "monolayer" / "run_epsilon_20_24iter.log"
HBN_LOG = ROOT / "epsilon_20" / "run_24iter_fresh.log"

def cfg_energy_grid(path):
    with path.open("rb") as f:
        cfg = tomllib.load(f)
    e = cfg["electron"]
    return np.linspace(e["energy_window_min"], e["energy_window_max"], e["energy_window_num"])

energies = cfg_energy_grid(HBN_CONFIG)
pure_energies = cfg_energy_grid(PURE_CONFIG)
assert len(energies) == len(pure_energies) and np.allclose(energies, pure_energies), "Energy grids differ"

hbar_eV_s = 6.582119569e-16
response_energies = energies - energies[0]
omega = response_energies / hbar_eV_s

def iteration_from_name(path):
    return int(path.stem.rsplit("_", 1)[1])

def iterations(out_dir, stem="electron_ldos"):
    return sorted(iteration_from_name(p) for p in out_dir.glob(f"{stem}_*.npy"))

pure_iterations = iterations(PURE)
hbn_iterations = iterations(HBN)
pure_it = pure_iterations[-1]
hbn_it = hbn_iterations[-1]

runs = {
    "pure MoS2": {"dir": PURE, "it": pure_it, "color": "tab:blue", "log": PURE_LOG},
    "MoS2+hBN eps=20": {"dir": HBN, "it": hbn_it, "color": "tab:green", "log": HBN_LOG},
}

def load_iter(run, stem, iteration):
    info = runs[run]
    return np.load(info["dir"] / f"{stem}_{iteration}.npy", mmap_mode="r")

def load(run, stem):
    return load_iter(run, stem, runs[run]["it"])

def maxabs_iter(run, stem, iteration):
    return float(np.nanmax(np.abs(load_iter(run, stem, iteration))))

def summed_trace(run, stem):
    return np.asarray(load(run, stem)).sum(axis=1)

def magnitude_trace(run, stem):
    return np.sum(np.abs(np.asarray(load(run, stem))), axis=1)

def plot_complex_trace(ax, x, y, label, *, color=None):
    real_line, = ax.plot(x, np.real(y), label=f"Re({label})", linewidth=2, alpha=0.85, color=color)
    imag_color = color if color is not None else real_line.get_color()
    ax.plot(x, np.imag(y), label=f"Im({label})", linewidth=2, alpha=0.75, color=imag_color, linestyle="--")

print(f"Pure MoS2: {PURE.resolve()} final iter {pure_it}")
print(f"MoS2+hBN: {HBN.resolve()} final iter {hbn_it}")
print(f"Energy grid: {energies[0]:.3f} to {energies[-1]:.3f} eV, {len(energies)} points")


## SCBA Convergence Comparison

These come from the run logs, so they compare the self-consistency behavior directly: maximum self-energy update and contact current difference per iteration.


In [ ]:
def parse_scba_log(path):
    text = Path(path).read_text(errors="replace")
    return {
        "updates": [float(x) for x in re.findall(r"Maximum Self-Energy Update: ([^\n]+)", text)],
        "current_diffs": [float(x) for x in re.findall(r"Contact Current Difference: ([^\n]+)", text)],
    }

log_data = {name: parse_scba_log(info["log"]) for name, info in runs.items()}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, info in runs.items():
    data = log_data[name]
    axes[0].plot(range(len(data["updates"])), data["updates"], marker="o", label=name, color=info["color"])
    axes[1].plot(range(len(data["current_diffs"])), data["current_diffs"], marker="o", label=name, color=info["color"])

axes[0].set_title("Maximum self-energy update")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Maximum update")
axes[0].legend()

axes[1].set_title("Contact current difference")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Difference")
axes[1].legend()

plt.tight_layout()

for name, data in log_data.items():
    print(f"{name}: final update={data['updates'][-1]:.6g}, final contact diff={data['current_diffs'][-1]:.6g}")


## Main Observable Trends Across Iterations

This is the comparison version of the `trend_stems` check: it tracks the max absolute value of the main saved observables over SCBA iterations.


In [ ]:
trend_stems = [
    "device_current",
    "electron_ldos",
    "p_retarded_density",
    "w_greater_density",
    "sigma_retarded_density",
]

fig, axes = plt.subplots(1, len(trend_stems), figsize=(17, 3.5), sharex=True)
for ax, stem in zip(axes, trend_stems):
    for name, info in runs.items():
        xs, ys = [], []
        for it in iterations(info["dir"], stem):
            xs.append(it)
            ys.append(maxabs_iter(name, stem, it))
        ax.plot(xs, ys, marker="o", linewidth=1.6, label=name, color=info["color"])
    ax.set_title(stem)
    ax.set_xlabel("Iteration")
    ax.set_ylabel("max |value|")
    ax.legend()
plt.tight_layout()


## Environment Dielectric Response

This uses the hBN export file `epsilon_environment_inverse_retarded.npy`. The plotted quantity is the average diagonal/trace of the environment inverse dielectric matrix, not the scalar input `epsilon_r`.


In [ ]:
eps_inv_path = ROOT / "epsilon_20" / "outputs_hbn_environment" / "environment" / "epsilon_environment_inverse_retarded.npy"

if eps_inv_path.exists():
    eps_inv = np.load(eps_inv_path, mmap_mode="r")
    eps_inv_trace = np.array([np.trace(eps_inv[i]) / eps_inv.shape[1] for i in range(eps_inv.shape[0])])
    eps_inv_diag_abs_mean = np.array([np.mean(np.abs(np.diag(eps_inv[i]))) for i in range(eps_inv.shape[0])])

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)
    axes[0].plot(response_energies, eps_inv_trace.real, label="Re trace avg")
    axes[0].plot(response_energies, eps_inv_trace.imag, label="Im trace avg", linestyle="--")
    axes[0].set_title("hBN environment epsilon_E^{-1}")
    axes[0].set_xlabel("Response energy omega = E - E_min (eV)")
    axes[0].set_ylabel("trace(epsilon_E^{-1}) / N")
    axes[0].legend()

    axes[1].plot(response_energies, eps_inv_diag_abs_mean, color="tab:purple")
    axes[1].set_title("mean |diag(epsilon_E^{-1})|")
    axes[1].set_xlabel("Response energy omega = E - E_min (eV)")
    axes[1].set_ylabel("Mean diagonal magnitude")

    plt.tight_layout()
    print(f"Loaded {eps_inv_path}")
    print(f"shape={eps_inv.shape}, maxabs={np.nanmax(np.abs(eps_inv)):.6g}")
else:
    print(f"Missing {eps_inv_path}")


## Final-Iteration Magnitude Summary


In [ ]:
summary_stems = ["device_current", "electron_ldos", "p_retarded_density", "w_greater_density", "sigma_retarded_density"]
for stem in summary_stems:
    print(f"\n{stem}")
    for name in runs:
        arr = load(name, stem)
        print(f"  {name:16s} iter {runs[name]['it']:2d} maxabs={np.nanmax(np.abs(arr)):.6g} finite={np.isfinite(arr).all()}")



## Currents vs Energy


In [ ]:
current_stems = ["i_left", "i_right", "i_meir-wingreen", "device_current"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for ax, stem in zip(axes.ravel(), current_stems):
    for name, info in runs.items():
        y = np.asarray(load(name, stem)).squeeze()
        ax.plot(energies, np.real(y), label=f"{name} iter {info['it']}", color=info["color"])
    ax.set_title(stem)
    ax.set_xlabel("Energy (eV)")
    ax.set_ylabel("Current")
    ax.legend()
plt.tight_layout()



## EDOS vs Energy


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for name, info in runs.items():
    plot_complex_trace(ax, energies, summed_trace(name, "electron_ldos"), f"{name} iter {info['it']}", color=info["color"])
ax.set_xlabel("Energy (eV)")
ax.set_ylabel("Sum electron LDOS")
ax.set_title("EDOS: pure MoS2 vs MoS2+hBN")
ax.legend()
plt.tight_layout()



## Retarded Polarization vs Angular Frequency


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for name, info in runs.items():
    plot_complex_trace(ax, response_energies, summed_trace(name, "p_retarded_density"), f"{name} iter {info['it']}", color=info["color"])
ax.set_xlabel("Response energy omega = E - E_min (eV)")
ax.set_ylabel("Sum retarded polarization density")
ax.set_title("Retarded polarization: pure MoS2 vs MoS2+hBN")
ax.legend()
plt.tight_layout()



## Screened Interaction and Retarded Self-Energy

`W_greater` is plotted as `sum |W_greater_density|` to avoid cancellation from signed orbital sums.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)

for name, info in runs.items():
    axes[0].plot(energies, magnitude_trace(name, "w_greater_density"), label=f"{name} iter {info['it']}", color=info["color"])
axes[0].set_title("sum |W_greater density|")
axes[0].set_xlabel("Energy / frequency grid (eV)")
axes[0].set_ylabel("Sum magnitude over orbitals")
axes[0].legend()

for name, info in runs.items():
    plot_complex_trace(axes[1], energies, summed_trace(name, "sigma_retarded_density"), f"{name} iter {info['it']}", color=info["color"])
axes[1].set_title("sum Sigma_retarded density")
axes[1].set_xlabel("Energy / frequency grid (eV)")
axes[1].set_ylabel("Sum over orbitals")
axes[1].legend()

plt.tight_layout()

